## Generating Time Series using a Conditional Variational Autoencoder

### Download FX Timeseries from Yahoo
Download daily historical data for 20 currency pairs from 2001-01-01 to 2021-01-01

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# tqdm
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
from empirical_tests import run_tests, plot_daily_returns, acf_plots_block, qq_plot
from metrics import calculate_scores, real_data_loading
from timeseries import log_returns, download_data, normalize, denormalize, TimeSeriesDataset

### Download 4 Real Timeseries from Yahoo
Download daily historical data for Apple Inc. (AAPL) stock, the S&P 500 Index, the USD/JPY FX spot rate, and the Gold commodity spot price from 2001-01-01 to 2021-01-01

In [ ]:
start_date = "2001-01-01"
end_date = "2021-01-01"

 # Single stock example: Apple Inc. (AAPL)
single_stock = download_data("AAPL", start_date=start_date, end_date=end_date)

# Stock index example: S&P 500 Index (^GSPC)
stock_index = download_data("^GSPC", start_date=start_date, end_date=end_date)

# FX spot example: USD/JPY (JPY=X)
fx_spot = download_data("JPY=X", start_date=start_date, end_date=end_date)

# Commodity spot example: Gold (GC=F)
commodity_spot = download_data("GC=F", start_date=start_date, end_date=end_date)

ts_lst = [single_stock['Close'].to_numpy(), stock_index['Close'].to_numpy(), fx_spot['Close'].to_numpy(), commodity_spot['Close'].to_numpy()]

single_stock_returns = log_returns(single_stock['Close'])
stock_index_returns = log_returns(stock_index['Close'])
fx_spot_returns = log_returns(fx_spot['Close'])
commodity_spot_returns = log_returns(commodity_spot['Close'])

lst_returns = [single_stock_returns, stock_index_returns, fx_spot_returns, commodity_spot_returns]
lst_labels = ['Apple', 'S&P 500', 'USDJPY', 'Gold']

In [ ]:
norm_log_ssr, mlr_ssr, slr_ssr = normalize(single_stock_returns)
norm_log_sir, mlr_sir, slr_sir = normalize(stock_index_returns)
norm_log_fxr, mlr_fxr, slr_fxr = normalize(fx_spot_returns)
norm_log_csr, mlr_csr, slr_csr = normalize(commodity_spot_returns)

In [ ]:
# Convert DataFrame to PyTorch Tensor
ssr_tensor = torch.tensor(norm_log_ssr, dtype=torch.float32)
sir_tensor = torch.tensor(norm_log_sir, dtype=torch.float32)
fxr_tensor = torch.tensor(norm_log_fxr, dtype=torch.float32)
csr_tensor = torch.tensor(norm_log_csr, dtype=torch.float32)

In [ ]:
window_size = 1
condition_size = 1

In [ ]:
# Create TensorDataset and DataLoader
ssr_dataset = TimeSeriesDataset(ssr_tensor, window_size, condition_size)
sir_dataset = TimeSeriesDataset(sir_tensor, window_size, condition_size)
fxr_dataset = TimeSeriesDataset(fxr_tensor, window_size, condition_size)
csr_dataset = TimeSeriesDataset(csr_tensor, window_size, condition_size)

In [ ]:
batch_size = 128
ssr_dataloader = DataLoader(ssr_dataset, batch_size=batch_size, shuffle=False)
sir_dataloader = DataLoader(sir_dataset, batch_size=batch_size, shuffle=False)
fxr_dataloader = DataLoader(fxr_dataset, batch_size=batch_size, shuffle=False)
csr_dataloader = DataLoader(csr_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)
real_lst = [ssr2, sir2, fxr2, csr2]

### Build the VAE model

In [ ]:
class CVAE(nn.Module):
    def __init__(self, input_size, hidden_size, latent_size, condition_size):
        super(CVAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size + condition_size, hidden_size),
            nn.LeakyReLU(),
            nn.Linear(hidden_size, latent_size * 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_size + condition_size, hidden_size),
            nn.LeakyReLU(),
            nn.Linear(hidden_size, input_size),
        )

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, condition):
        h = torch.cat([x, condition], -1)
        latent_params = self.encoder(h)
        mu, log_var = latent_params.chunk(2, dim=-1)
        z = self.reparameterize(mu, log_var)
        z_cond = torch.cat([z, condition], -1)
        return self.decoder(z_cond), mu, log_var

def train(model, dataloader, optimizer, device):
    model.train()
    train_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        recon_y, mu, log_var = model(y, x)
        recon_loss = nn.MSELoss()(recon_y, y)
        kld_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        loss = recon_loss + kld_loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    return train_loss / len(dataloader)

In [ ]:
learning_rate = 1e-3
num_epochs = 1000
input_size = 1
hidden_size = 64
latent_size = 8
sample_len = len(single_stock_returns)

In [ ]:
sim_returns = list()

### Train the Single Stock Model


In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
model = CVAE(input_size, hidden_size, latent_size, condition_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()
tqdm_epoch = trange(num_epochs)

for epoch in tqdm_epoch:
    train_loss = train(model, ssr_dataloader, optimizer, device)
    tqdm_epoch.set_description('Train Loss: {:4f}'.format(train_loss))
    torch.save(model.state_dict(), 'cvae_genckpt.pth')

In [ ]:
def sample(model, condition, num_samples, device):
    model.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, latent_size).to(device)
        condition = condition.to(device)
        z_cond = torch.cat([z, condition], -1)
        generated_samples = model.decoder(z_cond)
    return generated_samples.cpu()

In [ ]:
sample_lst = list()

In [ ]:
last = ssr_tensor[-1].reshape([1, 1])
last

In [ ]:
for i in range(sample_len):
    new_sample = sample(model, last, 1, device)
    last = new_sample
    sample_lst.append(new_sample.item())

In [ ]:
norm_sample_ssr = np.array(sample_lst)
sample_ssr = denormalize(norm_sample_ssr, mlr_ssr, slr_ssr)

In [ ]:
sim_returns.append(sample_ssr)

### Train Stock Index Model

In [ ]:
model2 = CVAE(input_size, hidden_size, latent_size, condition_size).to(device)
optimizer = optim.Adam(model2.parameters(), lr=learning_rate)

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()

In [ ]:
tqdm_epoch = trange(num_epochs)
for epoch in tqdm_epoch:
    train_loss = train(model2, sir_dataloader, optimizer, device)
    tqdm_epoch.set_description('Train Loss: {:4f}'.format(train_loss))
    torch.save(model2.state_dict(), 'cvae2_genckpt.pth')

In [ ]:
sample_lst = list()
last = sir_tensor[-1].reshape([1, 1])

for i in range(sample_len):
    new_sample = sample(model2, last, 1, device)
    last = new_sample
    sample_lst.append(new_sample.item())

In [ ]:
norm_sample_sir = np.array(sample_lst)
sample_sir = denormalize(norm_sample_sir, mlr_sir, slr_sir)
sim_returns.append(sample_sir)

### Train FX Model

In [ ]:
model3 = CVAE(input_size, hidden_size, latent_size, condition_size).to(device)
optimizer = optim.Adam(model3.parameters(), lr=learning_rate)

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()

In [ ]:
tqdm_epoch = trange(num_epochs)
for epoch in tqdm_epoch:
    train_loss = train(model3, fxr_dataloader, optimizer, device)
    tqdm_epoch.set_description('Train Loss: {:4f}'.format(train_loss))
    torch.save(model3.state_dict(), 'cvae3_genckpt.pth')

In [ ]:
sample_lst = list()
last = fxr_tensor[-1].reshape([1, 1])

for i in range(sample_len):
    new_sample = sample(model3, last, 1, device)
    last = new_sample
    sample_lst.append(new_sample.item())

In [ ]:
norm_sample_fxr = np.array(sample_lst)
sample_fxr = denormalize(norm_sample_fxr, mlr_fxr, slr_fxr)
sim_returns.append(sample_fxr)

### Train Commodity Model

In [ ]:
model4 = CVAE(input_size, hidden_size, latent_size, condition_size).to(device)
optimizer = optim.Adam(model4.parameters(), lr=learning_rate)

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()

In [ ]:
tqdm_epoch = trange(num_epochs)
for epoch in tqdm_epoch:
    train_loss = train(model4, csr_dataloader, optimizer, device)
    tqdm_epoch.set_description('Train Loss: {:4f}'.format(train_loss))
    torch.save(model4.state_dict(), 'cvae4_genckpt.pth')

In [ ]:
sample_lst = list()
last = fxr_tensor[-1].reshape([1, 1])

for i in range(sample_len):
    new_sample = sample(model4, last, 1, device)
    last = new_sample
    sample_lst.append(new_sample.item())

In [ ]:
norm_sample_csr = np.array(sample_lst)
sample_csr = denormalize(norm_sample_csr, mlr_csr, slr_csr)
sim_returns.append(sample_csr)

#### Daily Returns

In [ ]:
cvae_filename = 'CVAEReturns.png'
plot_daily_returns(sim_returns, stock_index.index[:-1], cvae_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
df = run_tests(sim_returns, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
cvae_filename = 'CVAEACFs.png'
acf_plots_block(sim_returns, lst_labels, cvae_filename)

#### QQ Plot

In [ ]:
def qq_plot(real_returns_list, sim_returns_list, save_path, num_cols=2, labels=None):
    num_plots = len(real_returns_list)
    num_rows = int(np.ceil(num_plots / num_cols))
    fig, axs = plt.subplots(num_rows, num_cols, figsize=(10, 10))
    quantiles = np.linspace(start=0, stop=1, num=500)
    for i in range(num_plots):
        row = i // num_cols
        col = i % num_cols
        if labels:
            axs[row, col].set_title(labels[i])
        else:
            axs[row, col].set_title(f'QQ plot {i+1}')
        y_quantiles = np.quantile(real_returns_list[i], quantiles, method='nearest')
        x_quantiles = np.quantile(sim_returns_list[i], quantiles, method='nearest')
    
        axs[row, col].plot(x_quantiles, y_quantiles, 'ko')
        axs[row, col].set_xlabel('Simulated Quantiles')
        axs[row, col].set_ylabel('Real Quantiles')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

In [ ]:
cvae_qq_filename = 'CVAEQQ.png'
qq_plot(lst_returns, sim_returns, cvae_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in sim_returns:
    syn_lst.append(real_data_loading(isym, 32))

In [ ]:
cvae_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
cvae_df

In [ ]:
cvae_df.to_latex()